In [1]:
# =============================================================
# CELL 1: Install & Setup
# =============================================================
# FIX: Pin transformers to 4.44.0 — version 5.x uses meta device
# init that breaks DNABERT-2's custom ALiBi tensor building.
!pip install transformers==4.44.0 torch scikit-learn tqdm einops accelerate

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import glob
import os
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {device}")
if device.type == 'cuda':
    print(f"Ekran Kartı: {torch.cuda.get_device_name(0)}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cihaz: cuda
Ekran Kartı: NVIDIA A100-SXM4-40GB


In [14]:
# =============================================================
# CELL 2: Config
# =============================================================
NORMAL_DIR = '/content/drive/MyDrive/DNA_Anomaly_Detection/raw_normal_batches/'
CANCER_DIR = '/content/drive/MyDrive/DNA_Anomaly_Detection/raw_cancer_batches/'

CHECKPOINT_DIR = '/content/drive/MyDrive/DNA_Anomaly_Detection/dnabert_checkpoints/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best_dnabert_model.pth')

BATCH_SIZE = 256  # A100 GPU
MAX_LEN = 128
EPOCHS = 3
LEARNING_RATE = 2e-5

In [15]:
# =============================================================
# CELL 3: Dataset, Tokenizer, Model Definition
# =============================================================
from transformers import AutoConfig, AutoModel, AutoTokenizer
from transformers.models.bert.configuration_bert import BertConfig
import torch.nn as nn
import torch
from torch.utils.data import Dataset

class GenomicDataset(Dataset):
    def __init__(self, normal_files, cancer_files, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.sequences = []
        self.labels = []

        print("Veriler RAM'e yükleniyor...")
        for f in normal_files:
            with open(f, 'r') as file:
                lines = file.read().splitlines()
                self.sequences.extend(lines)
                self.labels.extend([0] * len(lines))

        for f in cancer_files:
            with open(f, 'r') as file:
                lines = file.read().splitlines()
                self.sequences.extend(lines)
                self.labels.extend([1] * len(lines))

        print(f"Toplam {len(self.sequences)} DNA dizisi yüklendi.")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(seq, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_len)
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.float32)
        }

print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# ================================================================
# FIX #2: Patch DNABERT-2's flash_attn_triton.py
# ================================================================
# DNABERT-2 ships a custom flash_attn_triton.py that uses:
#   tl.dot(q, k, trans_b=True)
# But Triton >= 3.0 (shipped with PyTorch 2.10 on Colab) removed
# the trans_b parameter. We need to patch the cached file so it
# uses tl.dot(q, tl.trans(k)) instead.
#
# We also need to handle trans_a=True the same way.
# ================================================================
import pathlib, re

# Find the cached flash_attn_triton.py
cache_dir = pathlib.Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules"
triton_files = list(cache_dir.rglob("flash_attn_triton.py"))

for fpath in triton_files:
    text = fpath.read_text()
    original = text

    # Pattern 1: tl.dot(a, b, trans_b=True) -> tl.dot(a, tl.trans(b))
    text = re.sub(
        r'tl\.dot\(([^,]+),\s*([^,]+),\s*trans_b\s*=\s*True\)',
        r'tl.dot(\1, tl.trans(\2))',
        text
    )
    # Pattern 2: tl.dot(a, b, trans_a=True) -> tl.dot(tl.trans(a), b)
    text = re.sub(
        r'tl\.dot\(([^,]+),\s*([^,]+),\s*trans_a\s*=\s*True\)',
        r'tl.dot(tl.trans(\1), \2)',
        text
    )

    if text != original:
        fpath.write_text(text)
        print(f"Patched: {fpath}")
    else:
        print(f"Already patched or no changes needed: {fpath}")

print("Triton flash_attn patch tamamlandı!")

# ================================================================
# Model Definition
# ================================================================
class DNABERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        config = BertConfig.from_pretrained("zhihan1996/DNABERT-2-117M")

        self.dnabert = AutoModel.from_pretrained(
            "zhihan1996/DNABERT-2-117M",
            config=config,
            trust_remote_code=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.dnabert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding).squeeze(-1)

Tokenizer yükleniyor...
Already patched or no changes needed: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT_hyphen_2_hyphen_117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py
Already patched or no changes needed: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py
Triton flash_attn patch tamamlandı!


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [16]:
# =============================================================
# CELL 4: Load Data
# =============================================================
normal_files = glob.glob(NORMAL_DIR + '*.txt')
cancer_files = glob.glob(CANCER_DIR + '*.txt')

full_dataset = GenomicDataset(normal_files, cancer_files, tokenizer, max_len=MAX_LEN)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

print(f"Eğitim batch sayısı: {len(train_loader)}")
print(f"Doğrulama batch sayısı: {len(val_loader)}")

Veriler RAM'e yükleniyor...
Toplam 1800000 DNA dizisi yüklendi.
Eğitim batch sayısı: 5625
Doğrulama batch sayısı: 1407


In [31]:
# =============================================================
# EMERGENCY FIX: Nuke the broken flash_attn_triton.py completely
# and force-reload the patched version
# =============================================================
import pathlib, re, shutil, importlib, sys

# 1. Patch the file on disk (in case it wasn't patched yet)
cache_dir = pathlib.Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules"
triton_files = list(cache_dir.rglob("flash_attn_triton.py"))

for fpath in triton_files:
    text = fpath.read_text()
    # Replace trans_b=True → tl.trans(k)
    new_text = re.sub(
        r'tl\.dot\(([^,]+),\s*([^,]+),\s*trans_b\s*=\s*True\)',
        r'tl.dot(\1, tl.trans(\2))',
        text
    )
    # Replace trans_a=True → tl.trans(a)
    new_text = re.sub(
        r'tl\.dot\(([^,]+),\s*([^,]+),\s*trans_a\s*=\s*True\)',
        r'tl.dot(tl.trans(\1), \2)',
        new_text
    )
    fpath.write_text(new_text)
    print(f"Patched: {fpath}")

    # Verify it worked
    verify = fpath.read_text()
    remaining = verify.count('trans_b') + verify.count('trans_a')
    print(f"  Remaining trans_a/trans_b occurrences: {remaining}")

# 2. Clear Triton compilation cache
triton_cache = pathlib.Path.home() / ".triton" / "cache"
if triton_cache.exists():
    shutil.rmtree(triton_cache)
    print("Triton cache cleared.")

# 3. Force Python to forget the old loaded module
keys_to_remove = [k for k in sys.modules if "flash_attn_triton" in k]
for k in keys_to_remove:
    del sys.modules[k]
    print(f"Evicted from sys.modules: {k}")

# 4. Also evict bert_layers since it imported flash_attn_triton
keys_to_remove = [k for k in sys.modules if "bert_layers" in k or "bert_padding" in k]
for k in keys_to_remove:
    del sys.modules[k]
    print(f"Evicted from sys.modules: {k}")

print("\n✅ Done. Now run Cell 5.")

Patched: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT_hyphen_2_hyphen_117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py
  Remaining trans_a/trans_b occurrences: 0
Patched: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py
  Remaining trans_a/trans_b occurrences: 0
Evicted from sys.modules: transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers
Evicted from sys.modules: transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_padding

✅ Done. Now run Cell 5.


In [32]:
class DNABERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        config = BertConfig.from_pretrained("zhihan1996/DNABERT-2-117M")
        self.dnabert = AutoModel.from_pretrained(
            "zhihan1996/DNABERT-2-117M",
            config=config,
            trust_remote_code=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(768, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)       # <-- removed Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.dnabert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs[0][:, 0, :]
        return self.classifier(cls_embedding).squeeze(-1)

print("✅ Class redefined.")

✅ Class redefined.


In [33]:
# =============================================================
# NUCLEAR OPTION: Disable DNABERT-2's broken flash attention
# Force it to use standard PyTorch attention instead
# =============================================================
import pathlib, sys

cache_dir = pathlib.Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules"
bert_files = list(cache_dir.rglob("bert_layers.py"))

for fpath in bert_files:
    text = fpath.read_text()
    original = text

    # Replace the flash attention import with a flag that disables it
    # The bert_layers.py checks `if flash_attn_qkvpacked_func is not None`
    # to decide whether to use flash attention. Set it to None = use standard attention.
    text = text.replace(
        "from flash_attn_triton import flash_attn_qkvpacked_func",
        "flash_attn_qkvpacked_func = None  # PATCHED: disable broken triton flash attn"
    )
    # Also handle the other import style
    text = text.replace(
        "from .flash_attn_triton import flash_attn_qkvpacked_func",
        "flash_attn_qkvpacked_func = None  # PATCHED: disable broken triton flash attn"
    )

    if text != original:
        fpath.write_text(text)
        print(f"✅ Flash attention disabled in: {fpath}")
    else:
        print(f"Already patched: {fpath}")

# Evict all cached modules so they reload
keys_to_remove = [k for k in sys.modules if any(x in k for x in ["flash_attn", "bert_layers", "bert_padding"])]
for k in keys_to_remove:
    del sys.modules[k]
    print(f"Evicted: {k}")

import shutil
triton_cache = pathlib.Path.home() / ".triton" / "cache"
if triton_cache.exists():
    shutil.rmtree(triton_cache)

print("\n✅ Flash attention tamamen devre dışı. Cell 5'i çalıştır.")

Already patched: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT_hyphen_2_hyphen_117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py
Already patched: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/bert_layers.py

✅ Flash attention tamamen devre dışı. Cell 5'i çalıştır.


In [35]:
# =============================================================
# CELL 5: Training (float32, no autocast, no flash attn)
# =============================================================

model = DNABERTClassifier()
model = model.to(device)  # float32, no bf16

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

start_epoch = 0
best_val_loss = float('inf')

if os.path.exists(CHECKPOINT_PATH):
    print("Checkpoint bulundu! Yükleniyor...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    best_val_loss = checkpoint['best_val_loss']
    start_epoch = checkpoint['epoch'] + 1
    print(f"Eğitim {start_epoch + 1}. Epoch'tan devam edecek.")

print("-" * 50)
for epoch in range(start_epoch, EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS} Başlıyor...")

    model.train()
    total_train_loss = 0
    correct_train = 0

    train_loop = tqdm(train_loader, leave=True, position=0)
    for batch in train_loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        predictions = model(input_ids, attention_mask)
        loss = criterion(predictions, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()
        correct_train += ((predictions > 0.0) == labels.bool()).sum().item()
        train_loop.set_description(f"Train Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = correct_train / len(train_dataset)

    model.eval()
    total_val_loss = 0
    correct_val = 0

    print("Doğrulama yapılıyor...")
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            predictions = model(input_ids, attention_mask)
            loss = criterion(predictions, labels)

            total_val_loss += loss.item()
            correct_val += ((predictions > 0.0) == labels.bool()).sum().item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = correct_val / len(val_dataset)

    print(f"Epoch {epoch + 1} Sonuçları:")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    if avg_val_loss < best_val_loss:
        print("🌟 Yeni en iyi model! Drive'a kaydediliyor...")
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, CHECKPOINT_PATH)

print("\n" + "=" * 50)
print("Eğitim tamamlandı!")

Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--------------------------------------------------

Epoch 1/3 Başlıyor...


Train Loss: 0.6155: 100%|██████████| 5625/5625 [32:06<00:00,  2.92it/s]

Doğrulama yapılıyor...


Epoch 1 Sonuçları:
Train Loss: 0.6531 | Train Acc: 0.6259
Val Loss:   0.6465 | Val Acc:   0.6315
🌟 Yeni en iyi model! Drive'a kaydediliyor...

Epoch 2/3 Başlıyor...


...

Doğrulama yapılıyor...


Epoch 2 Sonuçları:
Train Loss: 0.6418 | Train Acc: 0.6368
Val Loss:   0.6434 | Val Acc:   0.6355
🌟 Yeni en iyi model! Drive'a kaydediliyor...

Epoch 3/3 Başlıyor...


Train Loss: 0.6208: 100%|██████████| 5625/5625 [31:50<00:00,  2.94it/s]

Doğrulama yapılıyor...


Epoch 3 Sonuçları:
Train Loss: 0.6305 | Train Acc: 0.6482
Val Loss:   0.6455 | Val Acc:   0.6325

Eğitim tamamlandı!
